# 08b - Regression versus violation classification

Change only the learning objective, keeping the same ordered **18 no-SA features** from notebook 08:
v0 without `skew_pred` and `log_alpha`, plus `src_edge_contrast`.

| Model | Training target | Learned score; lower is accepted |
| --- | --- | --- |
| Regression | `log(max(relative_error, 0.001))` | Predicted log-error |
| Classification | `1` if relative error **>0.25**, otherwise `0` | Probability score for class 1 |

All other training parameters, query-ID splits, training-only validation IDs and early-stopping
patience stay fixed. Regression uses its existing L2 loss; classification uses binary log loss.
There is no resampling, new class weighting, feature addition, or deep-learning model.

Default seeds **0, 1, 2** give three classifier fits and up to three regressor fits. Compatible
notebook-08 regressors are reused by provenance; their scores and all cutoffs are recomputed.
`P(class 1)<0.05` is **not** a 5% violation guarantee. Use calibration to select each model's cutoff.

This comparison is exploratory because the existing test queries have already been inspected.
Report coverage together with accepted/violating counts and the number of rates above 5%; do not
select settings from test outcomes. Reserve a new query workload for final confirmation.
This is offline quality evaluation, not actual SA bypass or end-to-end query latency.


In [0]:
%pip install lightgbm==4.6.0


## Settings

Use the same input roots and compute/package environment as 08. The input layout is
`data_root/<dataset>/{queries-Ans.txt,queries-meta.csv}` and
`sample_root/<dataset>/{rows.csv,src_*.txt,arm_<resolution>.txt}`.

Keep `seeds=0,1,2`, `tree_budgets=4000`, `epsilons=0.25`, `delta=0.05`, `grid_size=32`,
`confidence=0.95`, and `num_threads=2` for the controlled comparison. Exactly one positive tree
budget and the fixed 25% label boundary are supported. Hyperparameter experiments belong in a
separate training-validation study; nothing is selected automatically using test data.

`regression_run=auto` searches completed personal notebook-08 runs. It reuses a unique compatible
18-feature regressor per seed, and trains missing ones. `none` trains all six models. An explicit
run directory can provide any requested seeds it contains; an incompatible explicit run fails.
If automatic matches are ambiguous, the error lists compatible run directories by seed.
Replace `auto` in the `regression_run` widget above with the intended completed 08 run directory,
then rerun **Review the training plan** before training. Choose the intended source run, never
the one with the best test results. Set `none` to train both objectives from scratch instead.

`copy_inputs=true` snapshots inputs; `mlflow_enabled=true` logs to a personal experiment.
Outputs always go to a new personal directory. Databricks keeps old widget values: review them.
If `repo_root` changes, rerun the Settings/import cell before the plan cell.


In [0]:
STUDY = 'objective-compare'
NOTEBOOK_FILENAME = '08b_regression_vs_classification.ipynb'
MODEL_KINDS = ('regression', 'classification')
SOURCE_VARIANT = 'v0_no_skew_src_contrast'
SCORERS = ('learned', 'skew_align')
POLICIES = ('empirical', 'grid_empirical', 'ucb_grid')

import hashlib
import importlib
import importlib.metadata
import inspect
import json
import math
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import time
from uuid import uuid4

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score


def _repo_root():
    try:
        notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        root = notebook.split('/ml/notebooks/')[0]
        return root if root.startswith('/Workspace/') else '/Workspace' + root
    except Exception:
        return os.environ.get('REPO_ROOT', os.getcwd())


def _defaults():
    repo = _repo_root()
    output, experiment = '', ''
    parts = Path(repo).parts
    if 'Users' in parts and len(parts) > parts.index('Users') + 1:
        user = parts.index('Users') + 1
        output = str(Path(*parts[:user + 1]) / 'synopsislake-results')
        experiment = f'/Users/{parts[user]}/synopsislake-{STUDY}'
    volume = os.environ.get('VOLUME', '/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake')
    return dict(repo_root=repo, data_root=volume + '/data/ml', sample_root=volume + '/output/ml',
                output_root=output, experiment=experiment, dataset='authors_range', seeds='0,1,2',
                regression_run='auto', epsilons='0.25', delta='0.05', tree_budgets='4000', grid_size='32', confidence='0.95',
                num_threads='2', copy_inputs='true', mlflow_enabled='true')


DEFAULTS = _defaults()
try:
    for key, value in DEFAULTS.items():
        dbutils.widgets.text(key, value)
    get = dbutils.widgets.get
except NameError:
    def get(name):
        return os.environ.get(name.upper(), DEFAULTS[name])

repo = Path(get('repo_root')).resolve()
for filename in ('features.py', 'features_v1.py', 'gate_eval.py'):
    if not (repo / 'ml' / filename).is_file():
        raise FileNotFoundError(f'Check repo_root: missing existing module {repo / "ml" / filename}')
sys.path.insert(0, str(repo / 'ml'))
for name in ('features', 'features_v1', 'gate_eval'):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
from features import FEATURES, check_estimates
from features_v1 import FEATURES_V1, GROUPS, build_table_v1, check_source_estimates
from gate_eval import PARAMS, cutoff, fit, split_ids


## Fixed feature schema and class labels

Equality at 25% is class 0 (qualified); only a strictly greater error is class 1. Training and
inference use the same 18 columns in the original order, including no SA inputs. The separately
reported SA reference is not an input. Exact counts below 10 remain excluded by the existing
pipeline; zero/small-count queries and arbitrary range-query populations are not assessed.


In [0]:
def feature_names():
    result = [name for name in FEATURES if name not in GROUPS['skew']] + ['src_edge_contrast']
    if len(result) != 18 or {'skew_pred', 'log_alpha'} & set(result):
        raise ValueError('The controlled comparison requires the existing ordered 18 no-SA features')
    return result


def violation_labels(errors):
    values = np.asarray(errors, dtype=float)
    if values.ndim != 1 or not np.isfinite(values).all() or (values < 0).any():
        raise ValueError('Relative errors must be a finite nonnegative vector')
    return (values > 0.25).astype(np.int8)


## Input and split helpers

These reuse notebook 08's file formats, read-only snapshots, output protection, and query-ID splits.
No additional repository module is required.


In [0]:
def _bool(value):
    if isinstance(value, bool):
        return value
    if str(value).lower() not in ("true", "false"):
        raise ValueError("boolean settings must be true or false")
    return str(value).lower() == "true"


def _jsonable(value):
    if isinstance(value, dict):
        return {str(k): _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, Path):
        return str(value)
    return value


def _write_json(path, value):
    Path(path).write_text(json.dumps(_jsonable(value), indent=2, allow_nan=False), encoding="utf-8")


def _hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def validate_output_root(output_root, repo_root, data_root, sample_root):
    if not str(output_root).strip():
        raise ValueError("Set output_root to a personal results directory before running")
    output = Path(output_root).expanduser().resolve()
    if output.is_relative_to(Path("/Workspace/Shared")):
        raise ValueError("output_root must be personal, not /Workspace/Shared")
    for protected in (repo_root, data_root, sample_root):
        if output.is_relative_to(Path(protected).resolve()):
            raise ValueError("output_root must be outside the checkout and input directories")
    return output


def make_split_manifest(df, seed):
    train, cal, test = split_ids(df.qId, seed)
    early = set(sorted(train)[::10])
    return pd.DataFrame([
        {"seed": seed, "qId": int(qid), "split": split, "early_stopping": qid in early}
        for split, ids in (("train", train), ("calibration", cal), ("test", test))
        for qid in sorted(ids)
    ])


def _auc(error, score, eps):
    bad = np.asarray(error) > eps
    return float(roc_auc_score(bad, score)) if len(np.unique(bad)) == 2 else np.nan


def _inputs(data_dir, run_dir):
    files = [data_dir / "queries-Ans.txt", data_dir / "queries-meta.csv", run_dir / "rows.csv"]
    for path in files:
        if not path.is_file():
            raise FileNotFoundError(f"Missing input: {path}. Git clone does not create logger outputs.")
    sources = sorted(run_dir.glob("src_*.txt"))
    if not sources:
        raise FileNotFoundError(f"Missing source histogram exports: {run_dir}/src_*.txt")
    rows = pd.read_csv(run_dir / "rows.csv", usecols=["method", "lonRes"])
    resolutions = rows.loc[rows.method == "ARM", "lonRes"].unique()
    if not len(resolutions):
        raise ValueError("rows.csv has no ARM records")
    files += sources + [run_dir / f"arm_{int(res)}.txt" for res in sorted(resolutions)]
    for path in files:
        if not path.is_file():
            raise FileNotFoundError(f"Missing histogram export: {path}")
    return files


def _snapshot(files, data_dir, run_dir, destination, copy_inputs):
    manifest = []
    for source in files:
        before = source.stat()
        target = source
        if copy_inputs:
            kind = "data" if source.parent == data_dir else "run"
            target = destination / "inputs" / kind / source.name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, target)
        checksum = _hash(target)
        after = source.stat()
        if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
            raise RuntimeError(f"Input changed during snapshot: {source}")
        manifest.append(dict(source=str(source), used_path=str(target), size_bytes=before.st_size, sha256=checksum))
    return manifest


## Unchanged calibration methods

The next two code cells are copied verbatim from 08. All three policies remain unchanged:
`empirical`, `grid_empirical`, and `ucb_grid`, evaluated separately for every merge rate.
Training score quantiles fix candidates; calibration alone selects cutoffs; test only evaluates.

The simultaneous family is `2 model kinds * seeds * 1 tree budget * 1 epsilon * rates * 2 scorers * K`.
With 18 rates, three seeds, K=32 and confidence=0.95, it is still **6,912 candidates** and the same
per-candidate alpha as the default 08 run. The SA reference remains a repeated reference, not
independent evidence. Regression cutoffs are recalculated, not copied from an old run.

The UCB assumes independent calibration queries within a rate and an unchanged query population.
Rates/seeds need not be independent for the union allocation. Repeated query rows across rates
are never pooled into a binomial sample. Population bounds do not guarantee every finite test
fraction is below 5%. Infeasible UCB means reject all, with undefined accepted-answer risk.


In [0]:
"""Fixed-grid, calibration-only conditional-risk checks for notebook 08.

Each candidate cutoff must be fixed using training information only. Within a
merge rate, calibration observations must represent distinct independent query
samples from the intended unchanged query population. Repeated query rows from
multiple merge rates must not be pooled as independent binomial trials.

The caller supplies a preallocated per-candidate error budget. Bonferroni over
the entire predeclared family then permits choosing any certified candidate,
even if the candidate decisions or merge rates are dependent. The bound targets
population violation probability conditional on acceptance, not a guarantee that
every finite test batch has an observed violation fraction below delta.
"""

import numpy as np
import pandas as pd
from scipy.stats import beta


def _finite_vector(value, name, *, nonempty=False):
    try:
        array = np.asarray(value, dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must be a finite numeric vector") from error
    if array.ndim != 1 or (nonempty and len(array) == 0) or not np.isfinite(array).all():
        raise ValueError(f"{name} must be a finite one-dimensional vector"
                         + (" with at least one value" if nonempty else ""))
    return array


def training_threshold_grid(train_score, grid_size):
    """Return at most K finite cutoffs from TRAIN scores, with complete ties.

    No calibration/test scores or errors enter this function. K is the fixed
    maximum family size even when duplicate quantiles leave fewer cutoffs. The
    finite training maximum is a candidate; accept-all infinity is not added.
    Reject-all is implicit and needs no statistical certification.
    """
    if (isinstance(grid_size, (bool, np.bool_))
            or not isinstance(grid_size, (int, np.integer)) or grid_size < 2):
        raise ValueError("grid_size must be an integer >= 2")
    scores = _finite_vector(train_score, "train_score", nonempty=True)
    return np.unique(np.quantile(scores, np.linspace(0.0, 1.0, int(grid_size))))


def calibration_grid_candidates(cal_score, cal_error, candidates, epsilon, delta,
                                alpha_per_candidate):
    """Audit training-fixed cutoffs using one-sided Clopper-Pearson bounds.

    Risk means P(relative error > epsilon | score <= cutoff). Conditional on
    accepted sample size, the binomial calculation needs independent queries
    and a frozen model/cutoff under the population assumptions above. The
    supplied alpha must already account for all configured candidate tests.
    A cutoff with no accepted calibration observations is not certified.
    """
    score = _finite_vector(cal_score, "cal_score")
    error = _finite_vector(cal_error, "cal_error")
    thresholds = _finite_vector(candidates, "candidates", nonempty=True)
    if score.shape != error.shape:
        raise ValueError("cal_score and cal_error must have the same shape")
    if (error < 0).any():
        raise ValueError("cal_error must be nonnegative relative errors")
    try:
        epsilon, delta, alpha = float(epsilon), float(delta), float(alpha_per_candidate)
    except (TypeError, ValueError) as issue:
        raise ValueError("epsilon, delta and alpha must be finite numbers") from issue
    if not np.isfinite(epsilon) or epsilon <= 0:
        raise ValueError("epsilon must be finite and positive")
    if not np.isfinite(delta) or not 0 <= delta < 1:
        raise ValueError("delta must be finite and in [0, 1)")
    if not np.isfinite(alpha) or not 0 < alpha < 1:
        raise ValueError("alpha_per_candidate must be finite and in (0, 1)")
    rows = []
    for threshold in np.unique(thresholds):
        accepted = score <= threshold
        n = int(accepted.sum())
        k = int(((error > epsilon) & accepted).sum())
        observed = k / n if n else np.nan
        # isf(alpha, ...) equals ppf(1-alpha, ...) while avoiding loss of
        # precision when 1-alpha rounds to 1 for very small family budgets.
        upper = float(beta.isf(alpha, k + 1, n - k)) if n and k < n else (1.0 if n else np.nan)
        rows.append(dict(
            cutoff=float(threshold), calibration_accepted_count=n,
            calibration_violation_count=k, calibration_violation=observed,
            calibration_risk_upper=upper,
            calibration_certified=bool(n > 0 and np.isfinite(upper) and upper <= delta),
            alpha_per_candidate=alpha,
        ))
    return pd.DataFrame(rows)


def select_grid_cutoff(candidate_frame):
    """Choose the largest certified cutoff, or reject all with -infinity.

    All candidates were audited with simultaneous error allocation. Selection
    need not assume the conditional error rate increases with the cutoff.
    """
    required = {"cutoff", "calibration_certified"}
    if not isinstance(candidate_frame, pd.DataFrame) or not required <= set(candidate_frame.columns):
        raise ValueError("candidate_frame is missing cutoff/certification columns")
    values = _finite_vector(candidate_frame.cutoff.to_numpy(), "candidate_frame.cutoff")
    certified = candidate_frame.calibration_certified
    if certified.isna().any() or not certified.map(lambda x: isinstance(x, (bool, np.bool_))).all():
        raise ValueError("calibration_certified must contain booleans")
    selected = values[certified.to_numpy(dtype=bool)]
    return float(selected.max()) if len(selected) else -np.inf


In [0]:
def evaluate_rate_scores(train, cal, test, train_scores, cal_scores, test_scores,
                         epsilons, delta, alpha_each, grid_size):
    parts = (train, cal, test)
    score_sets = (train_scores, cal_scores, test_scores)
    rates = sorted(train.mergeRate.unique())
    if any(not len(part) or set(part.mergeRate.unique()) != set(rates) for part in parts):
        raise ValueError('Every cohort must contain the same nonempty merge-rate set')
    if any(part.duplicated(['qId', 'mergeRate']).any() for part in parts):
        raise ValueError('Per-rate binomial evaluation requires distinct query IDs')
    for part, scores in zip(parts, score_sets):
        if set(scores) != set(SCORERS):
            raise ValueError('Expected learned and skew_align score vectors')
        for score in scores.values():
            if np.asarray(score).shape != (len(part),) or not np.isfinite(score).all():
                raise ValueError('Scores must be finite vectors aligned with their cohort')
    details, audits = [], []
    for scorer in SCORERS:
        for epsilon in epsilons:
            for rate in rates:
                tr = train.mergeRate.to_numpy() == rate
                ca = cal.mergeRate.to_numpy() == rate
                te = test.mergeRate.to_numpy() == rate
                if not tr.any() or not ca.any() or not te.any():
                    raise ValueError('A rate has an empty train/calibration/test cohort')
                train_score = np.asarray(train_scores[scorer])[tr]
                cal_score = np.asarray(cal_scores[scorer])[ca]
                test_score = np.asarray(test_scores[scorer])[te]
                cal_error, test_error = cal.relErr.to_numpy()[ca], test.relErr.to_numpy()[te]
                grid = training_threshold_grid(train_score, grid_size)
                candidate = calibration_grid_candidates(cal_score, cal_error, grid,
                                                        epsilon, delta, alpha_each)
                candidate.insert(0, 'candidate_id', np.arange(len(candidate)))
                audits.append(candidate.assign(score=scorer, eps=epsilon, mergeRate=rate,
                                                training_count=int(tr.sum())))
                grid_ok = candidate[(candidate.calibration_accepted_count > 0)
                                    & (candidate.calibration_violation <= delta)]
                grid_empirical = float(grid_ok.cutoff.max()) if len(grid_ok) else -np.inf
                thresholds = {'empirical': float(cutoff(cal_score, cal_error, epsilon, delta)),
                              'grid_empirical': grid_empirical, 'ucb_grid': select_grid_cutoff(candidate)}
                for policy, threshold in thresholds.items():
                    accepted_cal, accepted_test = cal_score <= threshold, test_score <= threshold
                    n_cal = int(accepted_cal.sum())
                    k_cal = int(((cal_error > epsilon) & accepted_cal).sum())
                    n_test = int(accepted_test.sum())
                    k_test = int(((test_error > epsilon) & accepted_test).sum())
                    risk = k_test / n_test if n_test else np.nan
                    certified, upper = None, np.nan
                    if policy == 'ucb_grid':
                        certified = bool(math.isfinite(threshold))
                        if certified:
                            chosen = candidate.loc[candidate.cutoff == threshold].iloc[0]
                            upper = float(chosen.calibration_risk_upper)
                    details.append(dict(
                        score=scorer, policy=policy, eps=float(epsilon), scope='per_rate', mergeRate=float(rate),
                        cutoff=threshold if math.isfinite(threshold) else None,
                        accept_none=not math.isfinite(threshold), training_count=int(tr.sum()),
                        calibration_count=int(ca.sum()), calibration_accepted_count=n_cal,
                        calibration_violation_count=k_cal, calibration_violation=k_cal / n_cal if n_cal else np.nan,
                        calibration_risk_upper=upper, calibration_certified=certified,
                        alpha_per_candidate=alpha_each if policy == 'ucb_grid' else np.nan,
                        test_count=int(te.sum()), accepted_count=n_test, violation_count=k_test,
                        coverage=n_test / int(te.sum()), violation=risk,
                        quality_target_met=(risk <= delta) if n_test else None,
                        auc=_auc(test_error, test_score, epsilon),
                    ))
    gates = pd.DataFrame(details)
    summaries = []
    for (scorer, policy, epsilon), group in gates.groupby(['score', 'policy', 'eps'], sort=False):
        accepted, bad = int(group.accepted_count.sum()), int(group.violation_count.sum())
        summaries.append(dict(
            score=scorer, policy=policy, eps=epsilon, auc_per_rate=group.auc.mean(),
            coverage_per_rate=group.coverage.mean(), violation_per_rate=group.violation.mean(),
            violation_pooled_per_rate_gates=bad / accepted if accepted else np.nan,
            rates_with_acceptances=int((group.accepted_count > 0).sum()), rates_total=len(group),
            accepted_cases_per_rate=accepted, violating_cases_per_rate=bad,
            calibration_certified_rates=int(group.calibration_certified.eq(True).sum()),
            unique_test_queries=int(test.qId.nunique()), test_cases=len(test),
        ))
    return pd.DataFrame(summaries), gates, pd.concat(audits, ignore_index=True)


## Compatible regressor reuse and the training plan

Reuse checks fingerprints, existing feature/training code, packages, parameters, ordered features,
the log-error target, the saved regression Booster, and exact query cohorts. A complete source-08
model and its historical fit time are required. Current input snapshots are checked again before
training. Reused weights are copied into this study; all scores and policies are evaluated anew.
No source results are modified. A changed notebook/export cell alone does not invalidate a model:
the existing three training/feature modules and saved model schema must still match.


In [0]:
def _read_json(path):
    try:
        return json.loads(Path(path).read_text())
    except (OSError, ValueError) as error:
        raise ValueError(f'Cannot read JSON at {path}: {error}') from error


def _input_hashes(data, samples):
    result = {}
    for path in _inputs(data, samples):
        before = path.stat()
        digest = _hash(path)
        after = path.stat()
        if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
            raise ValueError(f'Input changed during preflight: {path}')
        role = 'data' if path.parent == data else 'run'
        key = role + '/' + path.name
        if key in result:
            raise ValueError(f'Duplicate input identity: {key}')
        result[key] = digest
    return result


def _manifest_hashes(records):
    result = {}
    for row in records:
        name = Path(row['source']).name
        role = 'data' if name in ('queries-Ans.txt', 'queries-meta.csv') else 'run'
        key = role + '/' + name
        if key in result:
            raise ValueError(f'Duplicate saved input: {key}')
        result[key] = row['sha256']
    return result


def _check_splits(splits, seed):
    if not {'seed','qId','split','early_stopping'}.issubset(splits):
        raise ValueError('Saved split file is missing required columns')
    selected = splits[splits.seed == seed].copy()
    if selected.empty or selected.qId.duplicated().any():
        raise ValueError('Missing or duplicate saved query IDs')
    expected = make_split_manifest(pd.DataFrame({'qId': selected.qId}), seed)
    columns = ['seed', 'qId', 'split', 'early_stopping']
    try:
        pd.testing.assert_frame_equal(selected[columns].sort_values('qId').reset_index(drop=True),
                                      expected[columns].sort_values('qId').reset_index(drop=True),
                                      check_dtype=False)
    except AssertionError as error:
        raise ValueError('Saved query split or early-stopping IDs differ') from error
    return selected


def _check_cohorts(path, split, rates):
    cohorts = pd.read_csv(path)
    required = ['qId', 'mergeRate', 'split', 'early_stopping']
    if not set(required).issubset(cohorts) or cohorts.duplicated(['qId', 'mergeRate']).any():
        raise ValueError('Missing or duplicate saved model cohorts')
    if len(cohorts) != len(split) * len(rates) or set(cohorts.mergeRate) != set(rates):
        raise ValueError('Saved cohorts have different rates or query counts')
    reference = split.drop(columns='seed').sort_values('qId').reset_index(drop=True)
    for rate in rates:
        actual = cohorts.loc[cohorts.mergeRate == rate, reference.columns].sort_values('qId').reset_index(drop=True)
        try:
            pd.testing.assert_frame_equal(actual, reference, check_dtype=False)
        except AssertionError as error:
            raise ValueError('Saved model cohorts differ from the common query split') from error


def _regression_source(folder, seed, budget, plan):
    folder = Path(folder).resolve()
    state = _read_json(folder / 'run.json')
    if state.get('status') != 'complete' or state.get('study') != 'no-skew':
        raise ValueError('Source must be a completed notebook-08 no-skew run')
    saved = state.get('config', {})
    if saved.get('dataset') != plan['config']['dataset']:
        raise ValueError('Source dataset differs')
    if (budget not in [int(value) for value in str(saved.get('tree_budgets','')).split(',')]
            or int(saved.get('num_threads',-1)) != int(plan['config']['num_threads'])):
        raise ValueError('Source configured tree budget or thread count differs')
    if state.get('feature_definitions', {}).get(SOURCE_VARIANT) != plan['feature_names']:
        raise ValueError('Source is not the ordered 18-feature no-SA contrast model')
    if state.get('package_versions') != plan['package_versions'] or state.get('python') != plan['python']:
        raise ValueError('Python or package versions differ')
    if any(state.get('code_sha256', {}).get(name) != digest for name, digest in plan['core_sha256'].items()):
        raise ValueError('Existing training/feature code differs')
    records = _read_json(folder / 'input_manifest.json')
    if _manifest_hashes(records) != plan['input_sha256']:
        raise ValueError('Source input fingerprints differ')
    if state.get('input_sha256') != plan['input_sha256']:
        raise ValueError('Source run and input manifest disagree')
    if _bool(saved.get('copy_inputs', True)):
        snapshot_root = (folder / 'inputs').resolve()
        for row in records:
            used = Path(row['used_path']).resolve()
            if not used.is_relative_to(snapshot_root) or _hash(used) != row['sha256']:
                raise ValueError('Source input snapshot is missing or changed')
    model_dir = folder / 'models' / SOURCE_VARIANT / f'trees_{budget}' / f'seed_{seed}'
    schema = _read_json(model_dir / 'schema.json')
    expected_params = dict(PARAMS, n_estimators=budget, n_jobs=int(plan['config']['num_threads']))
    if (schema.get('features') != plan['feature_names'] or schema.get('params') != expected_params
            or schema.get('seed') != seed or schema.get('tree_budget') != budget
            or schema.get('variant') != SOURCE_VARIANT or schema.get('min_exact') != 10
            or schema.get('label') != 'log(max(abs(estimate-truth)/truth, 0.001))'):
        raise ValueError('Saved regression schema or training parameters differ')
    booster = lgb.Booster(model_file=str(model_dir / 'model.txt'))
    if booster.feature_name() != plan['feature_names'] or booster.params.get('objective') != 'regression':
        raise ValueError('Saved Booster is not the expected ordered regression model')
    splits = _check_splits(pd.read_csv(folder / 'splits.csv'), seed)
    _check_cohorts(model_dir / 'cohorts.csv', splits, plan['rates'])
    result = pd.read_csv(model_dir / 'results.csv')
    selected = result[(result.score == 'learned') & (result.seed == seed)
                      & (result.variant == SOURCE_VARIANT) & (result.tree_budget == budget)]
    if selected.empty or not np.isfinite(selected.fit_seconds).all() or not np.allclose(selected.fit_seconds, selected.fit_seconds.iloc[0]):
        raise ValueError('Missing or inconsistent original fit-time metadata')
    return dict(run_dir=str(folder), model_path=str(model_dir / 'model.txt'),
                model_sha256=_hash(model_dir / 'model.txt'), source_fit_seconds=float(selected.fit_seconds.iloc[0]),
                best_iteration=int(schema['best_iteration']))


def plan_experiment(config):
    cfg = dict(config)
    cfg['model_kinds'] = ','.join(MODEL_KINDS)
    if Path(cfg['repo_root']).resolve() != repo:
        raise ValueError('repo_root changed; rerun the Settings/import cell')
    dataset = str(cfg['dataset']).strip()
    if not dataset or dataset in ('.', '..') or '/' in dataset or '\\' in dataset:
        raise ValueError('dataset must be one directory name')
    cfg['dataset'] = dataset
    validate_output_root(cfg['output_root'], cfg['repo_root'], cfg['data_root'], cfg['sample_root'])
    seeds = [int(value.strip()) for value in str(cfg['seeds']).split(',')]
    budgets = [int(value.strip()) for value in str(cfg['tree_budgets']).split(',')]
    epsilons = [float(value.strip()) for value in str(cfg['epsilons']).split(',')]
    delta, confidence = float(cfg['delta']), float(cfg['confidence'])
    grid_size, threads = int(cfg['grid_size']), int(cfg['num_threads'])
    if not seeds or len(set(seeds)) != len(seeds) or min(seeds) < 0:
        raise ValueError('seeds must be distinct nonnegative integers')
    if len(budgets) != 1 or budgets[0] < 1 or threads < 1:
        raise ValueError('Use one positive shared tree budget and a positive thread count')
    if epsilons != [0.25]:
        raise ValueError('This controlled comparison fixes the classification boundary at 0.25')
    if not 0 <= delta < 1 or not 0 < confidence < 1 or grid_size < 2:
        raise ValueError('Invalid risk target, confidence, or grid size')
    _bool(cfg['copy_inputs'])
    if _bool(cfg['mlflow_enabled']) and not str(cfg['experiment']).startswith('/Users/'):
        raise ValueError('Set a personal /Users/<account>/... MLflow experiment')
    data, samples = Path(cfg['data_root']).resolve() / dataset, Path(cfg['sample_root']).resolve() / dataset
    identity = _input_hashes(data, samples)
    metadata = pd.read_csv(samples / 'rows.csv', usecols=['method', 'mergeRate'])
    rates = sorted(metadata.loc[metadata.method == 'ARM', 'mergeRate'].unique())
    if not rates or not np.isfinite(rates).all():
        raise ValueError('Input metadata needs finite merge rates')
    family = len(MODEL_KINDS) * len(seeds) * len(budgets) * len(epsilons) * len(rates) * len(SCORERS) * grid_size
    plan = dict(config=cfg, model_kinds=list(MODEL_KINDS), seeds=seeds, tree_budgets=budgets,
                feature_names=feature_names(), epsilons=epsilons, delta=delta, confidence=confidence,
                grid_size=grid_size, rates=rates, family_hypotheses=family,
                alpha_per_candidate=(1-confidence)/family, input_sha256=identity,
                package_versions={name: importlib.metadata.version(name) for name in
                                  ('lightgbm', 'numpy', 'pandas', 'scipy', 'scikit-learn')},
                python=platform.python_version(), core_sha256={name: _hash(repo / 'ml' / name) for name in
                                                             ('features.py', 'features_v1.py', 'gate_eval.py')})
    choice = str(cfg['regression_run']).strip()
    explicit = choice.lower() not in ('auto', 'none')
    output = Path(cfg['output_root']).resolve()
    if explicit:
        candidate = Path(choice).expanduser().resolve()
        if not candidate.is_dir() or not candidate.is_relative_to(output):
            raise ValueError('An explicit regression_run must exist under the personal output_root')
        folders = [candidate]
    else:
        folders = [] if choice.lower() == 'none' else sorted(output.glob(dataset+'-no-skew-*'))
    selected = {str(seed): None for seed in seeds}
    compatible = {str(seed): [] for seed in seeds}
    hashes, diagnostics = {}, []
    for folder in folders:
        available = [seed for seed in seeds if (folder/'models'/SOURCE_VARIANT/f'trees_{budgets[0]}'/f'seed_{seed}').is_dir()]
        if explicit and not available:
            raise ValueError('Chosen run contains no requested 18-feature regression model')
        for seed in available:
            try:
                info = _regression_source(folder, seed, budgets[0], plan)
            except (OSError, ValueError, KeyError, TypeError, lgb.basic.LightGBMError) as error:
                if explicit:
                    raise ValueError(f'Incompatible chosen regressor seed {seed}: {error}') from error
                diagnostics.append(f'{folder.name}/seed_{seed}: {error}')
                continue
            compatible[str(seed)].append(str(folder.resolve()))
            if selected[str(seed)] is None:
                selected[str(seed)] = str(folder.resolve())
                hashes[str(seed)] = info['model_sha256']
    ambiguous = {seed: paths for seed, paths in compatible.items() if len(paths) > 1}
    if ambiguous:
        details = '\n'.join(f'seed {seed}:\n' + '\n'.join(f'  {path}' for path in paths)
                            for seed, paths in ambiguous.items())
        raise ValueError('Multiple compatible source regressors found.\n' + details + '\n'
                         'Replace auto in the regression_run widget above with ONE completed 08 run directory '
                         '(the directory containing run.json), then rerun Review the training plan. '
                         'Choose the intended source run without comparing test results. '
                         'Alternatively, set regression_run to none to train all models from scratch. '
                         'This training plan has not trained any models.')
    reuse = sum(value is not None for value in selected.values())
    plan.update(regression_sources=selected, regression_model_sha256=hashes, reuse_diagnostics=diagnostics,
                reuse_count=reuse, fit_count=len(MODEL_KINDS)*len(seeds)-reuse)
    return plan


## Train or load, then evaluate both objectives identically

The classifier uses the same training-only validation query IDs and patience 100 as the regressor.
Its objective-specific validation loss is binary log loss, versus regression L2. A one-class fitting
cohort fails clearly without changing the split. All classifier scores are `P(class 1)`.
Every reused regressor is loaded as a Booster and predicts the new snapshot again; old test metrics
and cutoffs are never substituted. Current and historical training costs are recorded separately.


In [0]:
def _parts(frame, seed):
    ids = split_ids(frame.qId, seed)
    parts = tuple(frame[frame.qId.isin(group)] for group in ids)
    if parts[0].qId.nunique() < 3 or any(not len(part) for part in parts):
        raise ValueError('Insufficient train/calibration/test queries; do not resplit to repair the study')
    return parts


def _score(model, kind, part, features, threads):
    if kind == 'classification':
        classes = list(model.classes_)
        if classes != [0, 1]:
            raise ValueError('Classifier must represent class 0 and class 1')
        values = model.predict_proba(part[features], num_threads=threads)[:, classes.index(1)]
    else:
        values = model.predict(part[features], num_threads=threads)
    if not np.isfinite(values).all():
        raise ValueError('Nonfinite prediction scores')
    if kind == 'classification' and ((values < 0).any() or (values > 1).any()):
        raise ValueError('Classifier scores must be probabilities for class 1')
    return dict(learned=np.asarray(values), skew_align=part.skew_pred.to_numpy())


def _fit_or_load(frame, kind, seed, budget, plan):
    features = plan['feature_names']
    if {'skew_pred', 'log_alpha'} & set(features):
        raise ValueError('SA model inputs are forbidden')
    train, cal, test = _parts(frame, seed)
    params = dict(PARAMS, n_estimators=budget, n_jobs=int(plan['config']['num_threads']))
    source = plan['regression_sources'][str(seed)] if kind == 'regression' else None
    tick = time.perf_counter()
    source_info = _regression_source(Path(source), seed, budget, plan) if source else None
    if source_info:
        if source_info['model_sha256'] != plan['regression_model_sha256'][str(seed)]:
            raise ValueError('Planned regression weights changed; rerun the training plan')
        model = lgb.Booster(model_file=source_info['model_path'])
        if _hash(Path(source_info['model_path'])) != source_info['model_sha256']:
            raise ValueError('Regression weights changed during model loading')
        load_seconds = time.perf_counter()-tick
        fit_seconds = 0.0
        best_iteration = source_info['best_iteration']
    else:
        load_seconds = 0.0
        if kind == 'regression':
            model = fit(train, features, seed, params=params)
        else:
            early_ids = set(sorted(train.qId.unique())[::10])
            validation, fitting = train[train.qId.isin(early_ids)], train[~train.qId.isin(early_ids)]
            labels = violation_labels(fitting.relErr)
            if set(labels) != {0, 1}:
                raise ValueError('Classifier fitting data have only one class; keep the split and inspect the dataset')
            model = lgb.LGBMClassifier(random_state=seed, **params)
            model.fit(fitting[features], labels, eval_set=[(validation[features], violation_labels(validation.relErr))],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        fit_seconds = time.perf_counter()-tick
        best_iteration = int(model.best_iteration_)
    tick = time.perf_counter()
    scores = [_score(model, kind, part, features, params['n_jobs']) for part in (train, cal, test)]
    return dict(model=model, train=train, cal=cal, test=test, scores=scores, params=params,
                fit_seconds=fit_seconds, model_load_seconds=load_seconds,
                prediction_seconds=time.perf_counter()-tick, best_iteration=best_iteration,
                source=source_info)


def _decision_metrics(summary, gates, test, scores, epsilon, delta):
    gates = gates.copy()
    for index, row in gates.iterrows():
        mask = test.mergeRate.to_numpy() == row.mergeRate
        bad = violation_labels(test.relErr.to_numpy()[mask]).astype(bool)
        score = np.asarray(scores[row.score])[mask]
        true_good = int((~bad).sum())
        accepted_good = int(row.accepted_count-row.violation_count)
        gates.loc[index, 'true_good_count'] = true_good
        gates.loc[index, 'good_accepted_count'] = accepted_good
        gates.loc[index, 'precision_good'] = accepted_good/row.accepted_count if row.accepted_count else np.nan
        gates.loc[index, 'good_recall'] = accepted_good/true_good if true_good else np.nan
        gates.loc[index, 'average_precision_bad'] = average_precision_score(bad, score) if 0 < bad.mean() < 1 else np.nan
        gates.loc[index, 'average_precision_good'] = average_precision_score(~bad, -score) if 0 < bad.mean() < 1 else np.nan
    gates['true_good_count'] = gates.true_good_count.astype(int)
    gates['good_accepted_count'] = gates.good_accepted_count.astype(int)
    summary = summary.copy()
    for index, row in summary.iterrows():
        group = gates[(gates.score == row.score) & (gates.policy == row.policy) & (gates.eps == row.eps)]
        good, accepted_good = int(group.true_good_count.sum()), int(group.good_accepted_count.sum())
        summary.loc[index, 'true_good_cases'] = good
        summary.loc[index, 'good_accepted_cases'] = accepted_good
        summary.loc[index, 'good_recall_cases'] = accepted_good/good if good else np.nan
        summary.loc[index, 'precision_good_cases'] = accepted_good/row.accepted_cases_per_rate if row.accepted_cases_per_rate else np.nan
        summary.loc[index, 'good_recall_per_rate'] = group.good_recall.mean()
        summary.loc[index, 'precision_good_per_rate'] = group.precision_good.mean()
        summary.loc[index, 'average_precision_bad'] = group.average_precision_bad.mean()
        summary.loc[index, 'average_precision_good'] = group.average_precision_good.mean()
        summary.loc[index, 'rates_above_target'] = int((group.violation > delta).sum())
    for column in ('true_good_cases', 'good_accepted_cases', 'rates_above_target'):
        summary[column] = summary[column].astype(int)
    return summary, gates


def _record_model(frame, kind, seed, budget, plan, folder):
    fitted = _fit_or_load(frame, kind, seed, budget, plan)
    train, cal, test = fitted['train'], fitted['cal'], fitted['test']
    tick = time.perf_counter()
    result, gates, candidates = evaluate_rate_scores(train, cal, test, *fitted['scores'],
        plan['epsilons'], plan['delta'], plan['alpha_per_candidate'], plan['grid_size'])
    result, gates = _decision_metrics(result, gates, test, fitted['scores'][2], 0.25, plan['delta'])
    evaluation_seconds = time.perf_counter()-tick
    folder.mkdir(parents=True, exist_ok=False)
    if fitted['source']:
        original = Path(fitted['source']['model_path'])
        shutil.copyfile(original, folder/'model.txt')
        if (_hash(original) != fitted['source']['model_sha256']
                or _hash(folder/'model.txt') != fitted['source']['model_sha256']):
            raise ValueError('Source regression model changed while copying')
    else:
        fitted['model'].booster_.save_model(str(folder/'model.txt'))
    label = ('log(max(abs(estimate-truth)/truth, 0.001))' if kind == 'regression'
             else '1 if relative_error > 0.25, otherwise 0')
    _write_json(folder/'schema.json', dict(model_kind=kind, objective='regression' if kind == 'regression' else 'binary',
        features=plan['feature_names'], seed=seed, tree_budget=budget, params=fitted['params'],
        best_iteration=fitted['best_iteration'], min_exact=10, label=label,
        class_labels=[0, 1] if kind == 'classification' else None,
        score='predicted log-error' if kind == 'regression' else 'P(class 1 = error > 0.25)',
        lower_score_is_better=True, forbidden_model_features=['skew_pred', 'log_alpha']))
    _write_json(folder/'cutoffs.json', gates[gates.score == 'learned'].to_dict('records'))
    for name, part, score in zip(('training_scores', 'calibration_predictions', 'test_predictions'),
                                  (train, cal, test), fitted['scores']):
        part[['qId', 'mergeRate', 'relErr']].assign(bad_label=violation_labels(part.relErr),
            score=score['learned'], skew_align=score['skew_align']).to_csv(folder/(name+'.csv'), index=False)
    early_ids = set(sorted(train.qId.unique())[::10])
    cohorts = pd.concat([part[['qId', 'mergeRate']].assign(split=name)
        for name, part in (('train', train), ('calibration', cal), ('test', test))], ignore_index=True)
    cohorts['early_stopping'] = (cohorts.split == 'train') & cohorts.qId.isin(early_ids)
    cohorts.to_csv(folder/'cohorts.csv', index=False)
    for table in (result, gates, candidates):
        table.insert(0, 'model_kind', kind)
        table.insert(1, 'tree_budget', budget)
        table.insert(2, 'seed', seed)
        table.insert(3, 'n_features', len(plan['feature_names']))
    result['fit_seconds'] = fitted['fit_seconds']
    result['current_fit_seconds'] = fitted['fit_seconds']
    result['source_fit_seconds'] = fitted['source']['source_fit_seconds'] if fitted['source'] else np.nan
    result['model_load_seconds'] = fitted['model_load_seconds']
    result['prediction_seconds'] = fitted['prediction_seconds']
    result['evaluation_seconds'] = evaluation_seconds
    result['best_iteration'] = fitted['best_iteration']
    result['reused'] = fitted['source'] is not None
    result.to_csv(folder/'results.csv', index=False)
    candidates.to_csv(folder/'threshold_candidates.csv', index=False)
    candidates[['score','eps','mergeRate','candidate_id','cutoff','training_count']].to_csv(folder/'training_threshold_grid.csv', index=False)
    booster = fitted['model'] if isinstance(fitted['model'], lgb.Booster) else fitted['model'].booster_
    importance = pd.DataFrame(dict(feature=plan['feature_names'], gain=booster.feature_importance('gain'),
                                   model_kind=kind, tree_budget=budget, seed=seed))
    importance.to_csv(folder/'importance.csv', index=False)
    origin = dict(model_kind=kind, seed=seed, tree_budget=budget, reused=fitted['source'] is not None,
                  source_run=fitted['source']['run_dir'] if fitted['source'] else None,
                  model_sha256=_hash(folder/'model.txt'), current_fit_seconds=fitted['fit_seconds'],
                  source_fit_seconds=fitted['source']['source_fit_seconds'] if fitted['source'] else None,
                  model_load_seconds=fitted['model_load_seconds'])
    _write_json(folder/'model_source.json', origin)
    return result, gates, candidates, importance, origin


def _log_mlflow(mlflow, folder, result, cfg, features, params):
    with mlflow.start_run(run_name=f"{cfg['dataset']}-{folder.parent.parent.name}-{folder.parent.name}-{folder.name}"):
        mlflow.log_params(dict(dataset=cfg['dataset'], features=','.join(features), **params))
        mlflow.set_tag('personal_result_dir', str(folder))
        for row in result.itertuples():
            for metric in ('auc_per_rate', 'coverage_per_rate', 'violation_per_rate', 'good_recall_cases'):
                value = float(getattr(row, metric))
                if math.isfinite(value):
                    mlflow.log_metric(f'{row.score}_{row.policy}_{metric}', value)
        mlflow.log_metric('current_fit_seconds', float(result.current_fit_seconds.iloc[0]))
        mlflow.log_artifacts(str(folder), artifact_path='model_evaluation')


## Summaries and paired comparisons

Use accepted/violating counts, good-answer precision/recall, defined-risk groups and groups above
5% alongside coverage. Zero acceptance has undefined precision/risk. `paired_deltas.csv` is
classification minus regression at the same seed/budget/scorer/policy. Higher coverage is not a win
if violations also increase. ROC-AUC and average precision are supporting ranking metrics.

Regression log-error and classifier probability have different units: do not compare probability
directly with actual relative error using MAE. No classifier probability cutoff is assumed safe.


In [0]:
def _tables(results, gates, destination, delta):
    keys = ['model_kind','tree_budget','score','policy','eps']
    metrics = ['auc_per_rate','average_precision_bad','average_precision_good','coverage_per_rate',
               'violation_per_rate','violation_pooled_per_rate_gates','good_recall_cases',
               'precision_good_cases','fit_seconds']
    grouped = results.groupby(keys)
    summary = grouped[metrics].agg(['mean','std'])
    summary.columns = [f'{metric}_{stat}' for metric,stat in summary.columns]
    summary = summary.reset_index().merge(grouped.seed.nunique().rename('n_seed').reset_index())
    for metric in metrics:
        summary = summary.merge(grouped[metric].count().rename(metric+'_n').reset_index())
    totals = grouped[['accepted_cases_per_rate','violating_cases_per_rate','good_accepted_cases','true_good_cases',
                      'rates_above_target','rates_with_acceptances','rates_total']].sum().reset_index()
    totals = totals.rename(columns={'rates_above_target':'test_gate_groups_above_target',
                                    'rates_with_acceptances':'defined_test_gate_groups',
                                    'rates_total':'total_test_gate_groups'})
    totals['case_weighted_violation'] = totals.violating_cases_per_rate/totals.accepted_cases_per_rate.where(totals.accepted_cases_per_rate>0)
    totals['case_weighted_good_recall'] = totals.good_accepted_cases/totals.true_good_cases.where(totals.true_good_cases>0)
    summary = summary.merge(totals,on=keys,validate='one_to_one')
    summary.to_csv(destination/'summary.csv',index=False)
    rate = gates.copy()
    rate['above_target'] = (rate.accepted_count>0)&(rate.violation>delta)
    rate.groupby(keys+['mergeRate']).agg(n_seed=('seed','nunique'),coverage_mean=('coverage','mean'),
        coverage_std=('coverage','std'),violation_mean=('violation','mean'),violation_std=('violation','std'),
        defined_risk_seeds=('violation','count'),above_target_seeds=('above_target','sum'),
        accepted_cases=('accepted_count','sum'),violating_cases=('violation_count','sum'),
        good_accepted_cases=('good_accepted_count','sum'),true_good_cases=('true_good_count','sum'),
        good_recall_mean=('good_recall','mean')).reset_index().to_csv(destination/'per_rate_summary.csv',index=False)
    compared = ['coverage_per_rate','violation_per_rate','auc_per_rate','average_precision_good',
                'violation_pooled_per_rate_gates','accepted_cases_per_rate','violating_cases_per_rate',
                'good_accepted_cases','good_recall_cases','rates_above_target']
    paired_keys = ['tree_budget','seed','score','policy','eps']
    reference = results[results.model_kind=='regression'][paired_keys+compared].rename(columns={m:'baseline_'+m for m in compared})
    paired = results[results.model_kind=='classification'].merge(reference,on=paired_keys,validate='one_to_one')
    paired['baseline_model_kind'] = 'regression'
    for metric in compared:paired['delta_'+metric] = paired[metric]-paired['baseline_'+metric]
    paired.to_csv(destination/'paired_deltas.csv',index=False)
    policy_keys = ['model_kind','tree_budget','seed','score','eps']
    empirical = results[results.policy=='empirical'][policy_keys+compared].rename(columns={m:'empirical_'+m for m in compared})
    policy = results[results.policy!='empirical'].merge(empirical,on=policy_keys,validate='many_to_one')
    policy['baseline_policy'] = 'empirical'
    for metric in compared:policy['delta_'+metric] = policy[metric]-policy['empirical_'+metric]
    policy.to_csv(destination/'calibration_deltas.csv',index=False)
    return summary


## Execute a unique saved study

All variants use the same snapshot and query cohorts. Feature gain is recorded per model, but its
raw magnitude has different loss units for regression and classification and must not be compared
as a cross-objective importance score. Report all model/policy outcomes, including failure to
certify or test violations. Reusing a historical fit is not evidence that regression trains faster.


In [0]:
def run_experiment(config, plan=None):
    started = time.perf_counter()
    tick = time.perf_counter()
    current = plan_experiment(config)
    preflight_seconds = time.perf_counter()-tick
    if plan is not None:
        for key in ('config','regression_sources','regression_model_sha256','input_sha256',
                    'core_sha256','package_versions','python','rates','family_hypotheses'):
            if current[key] != plan.get(key):
                raise ValueError(f'Training plan changed ({key}); rerun the plan cell')
    plan, cfg = current, current['config']
    mlflow = None
    if _bool(cfg['mlflow_enabled']):
        import mlflow
        mlflow.set_experiment(cfg['experiment'])
    destination = Path(cfg['output_root']).resolve()/f"{cfg['dataset']}-{STUDY}-{uuid4().hex[:12]}"
    destination.mkdir(parents=True,exist_ok=False)
    state = dict(status='running',study=STUDY,config=cfg,model_kinds=plan['model_kinds'],
        feature_definitions={kind:plan['feature_names'] for kind in MODEL_KINDS},model_sources=[],
        timings={'preflight_seconds':preflight_seconds},
        package_versions=plan['package_versions'],python=plan['python'],
        calibration_family={key:plan[key] for key in ('family_hypotheses','alpha_per_candidate','confidence',
                            'grid_size','model_kinds','seeds','tree_budgets','epsilons','rates')},
        calibration_assumptions=[
            'Training fixes every candidate cutoff; calibration alone selects cutoffs.',
            'Independent queries within each rate and an unchanged query population are assumed.',
            'The configured family covers both model kinds and both scorers; later searches are not included.',
            'Population bounds do not guarantee observed test risk on every finite batch.',
            'Truth below10 is excluded; repeated query rows across rates are not pooled binomial trials.'],
        notes=[
            'Only the target/objective changes: same18features, query IDs, shared parameters and patience100.',
            'Regression uses training-validation L2; classification uses binary log loss.',
            'Classifier probability is a score, not an automatic risk guarantee.',
            'Previously examined test queries make this exploratory; reserve a new workload for final validation.',
            'No automatic test-guided model selection; AUC alone does not establish success.',
            'Offline quality only; full source features and logged SA references are still read.',
            'Seed SDs are descriptive, not confidence intervals; queries repeat across seeds/rates.',
            'fit_seconds/current_fit_seconds count only this run; reused original fit times are recorded separately.'])
    try:
        data = Path(cfg['data_root']).resolve()/cfg['dataset']
        samples = Path(cfg['sample_root']).resolve()/cfg['dataset']
        tick = time.perf_counter()
        manifest = _snapshot(_inputs(data,samples),data,samples,destination,_bool(cfg['copy_inputs']))
        _write_json(destination/'input_manifest.json',manifest)
        state['input_sha256'] = _manifest_hashes(manifest)
        if state['input_sha256'] != plan['input_sha256']:
            raise ValueError('Inputs changed after preflight; no models were trained')
        state['timings']['input_preparation_seconds'] = time.perf_counter()-tick
        if _bool(cfg['copy_inputs']):data,samples = destination/'inputs/data',destination/'inputs/run'
        tick = time.perf_counter()
        frame = build_table_v1(str(samples),str(data))
        if not len(frame) or frame.duplicated(['qId','mergeRate']).any():
            raise ValueError('Missing eligible rows or duplicate query/rate pairs')
        if set(frame.mergeRate)!=set(plan['rates']):
            raise ValueError('Actual rates differ from the predeclared family')
        if not np.isfinite(frame[FEATURES_V1+['label','relErr','mergeRate']].to_numpy(float)).all():
            raise ValueError('Nonfinite features or labels')
        state.update(rows=len(frame),unique_queries=int(frame.qId.nunique()),
                     dropped_small_exact=frame.attrs.get('dropped_small_exact',0),
                     estimate_gaps=dict(merged=check_estimates(frame),sources=check_source_estimates(frame)))
        state['timings']['feature_build_seconds'] = time.perf_counter()-tick
        splits = pd.concat([make_split_manifest(frame,seed) for seed in plan['seeds']],ignore_index=True)
        splits.to_csv(destination/'splits.csv',index=False)
        # Check every reused cohort against the actual current snapshot before ANY model fit.
        for seed in plan['seeds']:
            prior = plan['regression_sources'][str(seed)]
            if prior:
                old = _check_splits(pd.read_csv(Path(prior)/'splits.csv'),seed)
                expected = splits[splits.seed==seed].sort_values('qId').reset_index(drop=True)
                try:
                    pd.testing.assert_frame_equal(old.sort_values('qId').reset_index(drop=True),expected,check_dtype=False)
                except AssertionError as error:
                    raise ValueError('Source regression eligible query cohort differs from current inputs') from error
                _check_cohorts(Path(prior)/'models'/SOURCE_VARIANT/f"trees_{plan['tree_budgets'][0]}"/f'seed_{seed}'/'cohorts.csv',
                               expected,plan['rates'])
        state['code_sha256'] = dict(plan['core_sha256'])
        state['inline_helper_sha256'] = {}
        for function in (feature_names,violation_labels,_snapshot,training_threshold_grid,
                         calibration_grid_candidates,select_grid_cutoff,evaluate_rate_scores,
                         _regression_source,_fit_or_load,_record_model,_tables,plan_experiment,run_experiment):
            try:state['inline_helper_sha256'][function.__name__] = hashlib.sha256(inspect.getsource(function).encode()).hexdigest()
            except (OSError,TypeError) as error:state.setdefault('inline_helper_hash_unavailable',{})[function.__name__]=type(error).__name__
        try:state['code_sha256'][NOTEBOOK_FILENAME]=_hash(repo/'ml/notebooks'/NOTEBOOK_FILENAME)
        except OSError as error:state['notebook_hash_unavailable']=type(error).__name__
        try:
            state['git_commit']=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
            state['git_dirty']=bool(subprocess.check_output(['git','-C',str(repo),'status','--porcelain'],text=True,stderr=subprocess.DEVNULL))
        except (OSError,subprocess.CalledProcessError):state['git_commit']=None
        _write_json(destination/'run.json',state)
        results,gates,candidates,importances=[],[],[],[]
        budget=plan['tree_budgets'][0]
        for kind in MODEL_KINDS:
            for seed in plan['seeds']:
                folder=destination/'models'/kind/f'trees_{budget}'/f'seed_{seed}'
                result,detail,candidate,importance,origin=_record_model(frame,kind,seed,budget,plan,folder)
                results.append(result);gates.append(detail);candidates.append(candidate);importances.append(importance)
                state['model_sources'].append(origin)
                if mlflow is not None:
                    _log_mlflow(mlflow,folder,result,cfg,plan['feature_names'],dict(PARAMS,n_estimators=budget,n_jobs=int(cfg['num_threads'])))
                print(f"{kind} seed={seed}: {'reused' if origin['reused'] else 'trained'}; saved {folder}")
                _write_json(destination/'run.json',state)
        combined,details=pd.concat(results,ignore_index=True),pd.concat(gates,ignore_index=True)
        combined.to_csv(destination/'by_seed.csv',index=False)
        details.to_csv(destination/'gate_details.csv',index=False)
        pd.concat(candidates,ignore_index=True).to_csv(destination/'threshold_candidates.csv',index=False)
        pd.concat(importances,ignore_index=True).to_csv(destination/'importance.csv',index=False)
        _tables(combined,details,destination,plan['delta'])
        state['timings'].update(current_fit_seconds=sum(row['current_fit_seconds'] for row in state['model_sources']),
                                experiment_wall_seconds=time.perf_counter()-started)
        state['status']='complete'
        _write_json(destination/'run.json',state)
        print('COMPLETE:',destination)
        return destination
    except Exception as error:
        state.update(status='failed',error=f'{type(error).__name__}: {error}')
        _write_json(destination/'run.json',state)
        raise


## Review the training plan

Expect **3 new fits** when all prior regressors match, or **6 fits** when none match. Missing prior
seeds are trained. All policies share the same fitted models. The default 18-rate comparison still
allocates confidence over 6,912 candidates; no calibration method or confidence budget is changed.


In [0]:
CONFIG = {key:get(key) for key in DEFAULTS}
PLAN = plan_experiment(CONFIG)
print('Repo:',repo)
print('Read-only input roots:',CONFIG['data_root'],CONFIG['sample_root'])
print('Personal output root:',CONFIG['output_root'])
print('Features:',len(PLAN['feature_names']),PLAN['feature_names'])
print('Models:',PLAN['model_kinds'],'seeds:',PLAN['seeds'],'tree budget:',PLAN['tree_budgets'])
print('New fits:',PLAN['fit_count'],'reused regressors:',PLAN['reuse_count'])
for seed,path in PLAN['regression_sources'].items():print(f'Regression seed {seed}:',path or 'train a new regressor')
for message in PLAN['reuse_diagnostics'][:5]:print('Skipped incompatible source:',message)
print('Rates:',len(PLAN['rates']),'candidate family:',PLAN['family_hypotheses'],'alpha per candidate:',PLAN['alpha_per_candidate'])


## Train and evaluate

Missing files, incompatible explicit sources, or changed settings fail clearly. No synthetic data
are substituted. Re-running creates a new directory and leaves all previous results unchanged.


In [0]:
if {key:get(key) for key in DEFAULTS} != CONFIG:
    raise ValueError('Settings changed; rerun Review the training plan before training')
result_dir = run_experiment(CONFIG,plan=PLAN)
print('Results for this study:',result_dir)


## Read and share results

- `summary.csv`: means and sample SDs across seeds, valid-risk counts, accepted/violating cases,
  good-answer recall, and counts of defined/total/above-target test gate groups. A zero-acceptance
  policy has undefined risk, not a successful 0% violation result.
- `by_seed.csv`, `gate_details.csv`, `per_rate_summary.csv`: individual rates, sample counts,
  good precision/recall, calibrated cutoffs and observed test outcomes.
- `paired_deltas.csv`: classification minus regression, paired at the same seed/budget/scorer/policy.
  Positive coverage or recall alone is insufficient if observed violations also increase.
- `calibration_deltas.csv`: each grid policy minus empirical for the same model and scorer.
- `threshold_candidates.csv`: complete unchanged calibration candidate audit; it does not select on test.
- `models/<model_kind>/trees_<budget>/seed_<seed>/`: Booster, ordered schema, current cutoffs, query
  cohorts, training/calibration/test scores, importance and trained/reused source information.
- `run.json`, `input_manifest.json`, `splits.csv`: status, sources, hashes, settings, assumptions and timing.

Require `status=complete`. Reused original fit time is recorded separately from current fit cost;
do not interpret zero current regression fit time as an algorithmic training speedup.
SA appears under both objectives as the same reference, not additional independent evidence.
Seed SDs are descriptive, not confidence intervals. No stable 5% test-risk or latency claim is made.

Keep this comparison exploratory. Do not choose a model, confidence, or cutoff by repeatedly checking
this test set. Fix the method before evaluating a new reserved workload. A failed or all-rejecting
classifier/UCB result is a valid experimental outcome, not a reason to change splits silently.


In [0]:
metadata = json.loads((result_dir/'run.json').read_text())
if metadata['status'] != 'complete':raise ValueError('Study is incomplete; inspect run.json')
summary = pd.read_csv(result_dir/'summary.csv')
deltas = pd.read_csv(result_dir/'paired_deltas.csv')
try:
    display(summary)
    display(deltas)
except NameError:
    print(summary.to_string(index=False))
    print(deltas.to_string(index=False))
print('Result files:',result_dir)
print('Share summary.csv, by_seed.csv, gate_details.csv, paired_deltas.csv, per_rate_summary.csv and run.json for analysis.')


In [0]:
# Export saved per-query test predictions. Run only this cell; no model is trained.
# Leave blank to use result_dir from this session. After a restart, paste the
# completed run directory into EXPORT_RUN_DIR; earlier training cells are not needed.
EXPORT_RUN_DIR = ""

import csv as _export_csv
import json as _export_json
from pathlib import Path as _ExportPath
from uuid import uuid4 as _export_uuid4
from zipfile import ZipFile as _ExportZipFile, ZIP_DEFLATED as _EXPORT_DEFLATED

_export_source = EXPORT_RUN_DIR.strip() or globals().get("result_dir")
if not _export_source:
    raise ValueError("Set EXPORT_RUN_DIR to the completed 08b result directory, then rerun only this cell.")
_export_root = _ExportPath(_export_source).expanduser().resolve()
_export_run_file = _export_root / "run.json"
if not _export_run_file.is_file():
    raise FileNotFoundError(f"No run.json in {_export_root}; check EXPORT_RUN_DIR.")
_export_metadata = _export_json.loads(_export_run_file.read_text())
if _export_metadata.get("status") != "complete" or _export_metadata.get("study") != "objective-compare":
    raise ValueError("Export requires a completed notebook 08b objective-comparison run.")
_export_config = _export_metadata["config"]
_export_variants = [name.strip() for name in _export_config["model_kinds"].split(",")]
_export_budgets = [int(value.strip()) for value in str(_export_config["tree_budgets"]).split(",")]
_export_seeds = [int(value.strip()) for value in str(_export_config["seeds"]).split(",")]
_export_predictions = []
for _export_variant in _export_variants:
    for _export_budget in _export_budgets:
        for _export_seed in _export_seeds:
            _export_file = (_export_root / "models" / _export_variant / f"trees_{_export_budget}"
                            / f"seed_{_export_seed}" / "test_predictions.csv").resolve()
            if not _export_file.is_relative_to(_export_root):
                raise ValueError("A prediction path is outside the run directory.")
            if not _export_file.is_file():
                raise FileNotFoundError(f"Missing saved predictions: {_export_file}")
            with _export_file.open(newline="") as _export_stream:
                _export_columns = next(_export_csv.reader(_export_stream), [])
            if not {"qId", "mergeRate", "relErr", "score", "skew_align"}.issubset(_export_columns):
                raise ValueError(f"Missing required prediction columns in {_export_file}")
            _export_predictions.append(_export_file)
if not _export_predictions or len(set(_export_predictions)) != len(_export_predictions):
    raise ValueError("Run configuration has missing or duplicate model settings.")
_export_files = _export_predictions + [_export_run_file, _export_root / "gate_details.csv"]
for _export_file in _export_files:
    if not _export_file.is_file():
        raise FileNotFoundError(f"Missing diagnostic metadata: {_export_file}")

# Preserve model/budget/seed folders so the CSV filenames remain distinguishable.
# Include run and gate metadata for matching cutoffs; omit models and raw inputs.
_export_bundle = _export_root / f"08b_predictions_{_export_uuid4().hex[:8]}.zip"
with _ExportZipFile(_export_bundle, "x", compression=_EXPORT_DEFLATED) as _export_archive:
    for _export_file in _export_files:
        _export_archive.write(_export_file, arcname=_export_file.relative_to(_export_root).as_posix())
print(f"Packaged {len(_export_predictions)} prediction CSVs plus run/gate metadata.")
print("Run directory:", _export_root)
print("Download and upload this ZIP:", _export_bundle)
